# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [4]:
!pip install pdfplumber

  Using cached cffi-2.0.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 487.3 kB/s  0:00:13m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 155.2 kB/s  0:00:34m0:00:0100:03
Using cached cffi-2.0.0-cp314-cp314-macosx_11_0_arm64.whl (181 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 254.7 kB/s  0:00:18 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 163.2 kB/s  0:00:20m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [pdfplumber]7 [pdfplumber]x]


In [7]:
!pip install python-dotenv

In [11]:
#Import the necessary libs
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from dotenv import load_dotenv
import chromadb

In [9]:
# TODO: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [17]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [25]:
# Create retrieve_game tool
# It should use chroma client and collection created
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

@tool
def retrieve_game(query):
    response = collection.query(
        query_texts = [query], # send the text query to Chroma
        n_results = 5
    )

    # response["metadatas"][0] gives the list of matched game records from query
    return response["metadatas"][0]

#### Evaluate Retrieval Tool

In [33]:
# Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

@tool
def evaluate_retrieval(question, retrieved_docs):
    prompt = f""" 
    Your task is to evaluate whether the retrieved documents are enough to answer the user's question.

    User question:
    {question}

    Retrieved documents:
    {retrieved_docs}

    Decide:
    1. Are the documents useful enough to answer the question?
    2. Explain why or why not.
    """
    llm = LLM(model="gpt-4o-mini", temperature=0.0)
    messages = [SystemMessage(content="Respond only in JSON: {useful: bool, description: str}"), UserMessage(content=prompt)]
    response = llm.invoke(messages)
    import json; result = json.loads(response.content)
    return result

    

#### Game Web Search Tool

In [26]:
# Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(question):
    response = tavily_client.search(query=question)
    return response["results"]

### Agent

In [ ]:
class AgentState(TypedDict):
    user_query: str
    instructions: str
    messages: list
    current_tool_calls: list | None


llm = LLM(model="gpt-4o-mini", temperature=0.3, tools=tools)
response = llm.invoke(state["messages"])


In [27]:
# Create Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools developed

agent = Agent(
    model_name="gpt-4o-mini",
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    instructions="""
    You are UdaPlay, an AI research agent for the video game industry.

    When a user asks a question:
    1. Use retrieve_game to search the internal game database.
    2. Use evaluate_retrieval to decide whether the retrieved documents are sufficient.
    3. If the retrieved documents are not sufficient, use game_web_search.
    4. Provide a clear, structured final answer.
    5. Mention the source of the information you used.
    """
)


In [32]:
import uuid
session_id = uuid.uuid4().hex

In [34]:
# Invoke agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
    "What platform was that last game you mentioned for?"
]

for i, question in enumerate(questions):
    run = agent.invoke(question, session_id=session_id)
    final_state = run.get_final_state()
    answer = final_state["messages"][-1].content

    print(f"Question: {question}")
    print(f"Answer: {answer}")
    print("-" * 50)


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Question: When Pokémon Gold and Silver was released?
Answer: **Pokémon Gold and Silver** were released in Japan on **November 21, 1999**. In North America, the release date was **October 15, 2000**, and it was later released in Europe on **April 6, 2001**. 

These games are notable for being the first titles in the Pokémon series designed for the Game Boy Color, introducing new regions, Pokémon, and gameplay mechanics.

**Sources:**
1. [Wikipedia](https://en.wikipedia.org/wiki/Pok%C3%A9mon_Gold_and_Silver)
2. [Reddit](https://www.reddit.com/r/Gameboy/comme

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes

Suggestions to Make Your Project Stand Out
Personalize the Dataset: Add extra games, companies, or platforms to the dataset and demonstrate richer queries.

Advanced Memory: Implement persistent long-term memory so the agent “learns” from web searches.
Structured Output: Return answers in both natural language and structured JSON for easy integration.

Visualization: Create a dashboard or visualization of the agent’s retrieval process or knowledge base.

Custom Tools: Add extra tools, such as sentiment analysis of game reviews or trending games detection.